In [ ]:
#|default_exp hyperliquid

In [ ]:
#|hide
%load_ext autoreload
%autoreload 2

## Hyperliquid Refactored

This notebook refactors the original `hyperliquid.ipynb` to use the `DataSource` abstract base class and `polars` instead of `pandas`.

In [ ]:
#| export

from token_data.datasource import DataSource
from pathlib import Path
from typing import List
import polars as pl
from hyperliquid.info import Info
from hyperliquid.utils import constants
import logging
from datetime import datetime, timedelta
import eth_account
import json

In [ ]:
#| export

class HyperliquidDataSource(DataSource):
    """A data source for Hyperliquid."""

    def __init__(self, data_folder: Path, config_path: str = '../config_hyperliquid.json'):
        super().__init__(data_folder)
        self.info = self._setup(config_path)

    def _setup(self, config_path: str):
        with open(config_path) as f:
            config = json.load(f)
        account = eth_account.Account.from_key(config["secret_key"])
        address = config.get("account_address") or account.address
        logging.info(f"Running with account address: {address}")
        return Info(constants.MAINNET_API_URL, skip_ws=True)

    def get_tokens(self) -> List[str]:
        """Returns a list of available tokens."""
        meta = self.info.meta()
        return [coin['name'] for coin in meta['universe']]

    def get_prices(self, token: str, start_date: str, end_date: str, time_interval: str = '1h') -> pl.DataFrame:
        """Returns a DataFrame of prices for a given token."""
        start_time = int(datetime.fromisoformat(start_date.replace('Z', '')).timestamp() * 1000)
        end_time = int(datetime.fromisoformat(end_date.replace('Z', '')).timestamp() * 1000)
        candles = self.info.candles_snapshot(token, time_interval, start_time, end_time)
        if not candles:
            return pl.DataFrame()
        
        df = pl.DataFrame(candles)
        df = df.with_columns([
            pl.from_epoch(pl.col('t'), time_unit='ms').alias('datetime'),
            pl.lit(token).alias('token')
        ])
        return df.select(['datetime', 'o', 'h', 'l', 'c', 'v', 'token']).rename({'o': 'open', 'h': 'high', 'l': 'low', 'c': 'close', 'v': 'volume'})


### Tests

In [ ]:
import unittest
import shutil

class TestHyperliquidDataSource(unittest.TestCase):
    
    def setUp(self):
        self.data_folder = Path('./test_data')
        self.data_folder.mkdir(exist_ok=True)
        config = {'secret_key': '0x0000000000000000000000000000000000000000000000000000000000000001', 'account_address': '0x0000000000000000000000000000000000000000'}
        with open('config.json', 'w') as f:
            json.dump(config, f)
        self.hyperliquid = HyperliquidDataSource(self.data_folder, 'config.json')

    def tearDown(self):
        shutil.rmtree(self.data_folder)
        Path('config.json').unlink()

    def test_get_tokens(self):
        tokens = self.hyperliquid.get_tokens()
        self.assertIsInstance(tokens, list)
        self.assertGreater(len(tokens), 0)
        self.assertIn('BTC', tokens)

    def test_get_prices(self):
        end_date = datetime.now()
        start_date = end_date - timedelta(days=1)
        df = self.hyperliquid.get_prices('BTC', start_date.isoformat(), end_date.isoformat())
        self.assertIsInstance(df, pl.DataFrame)
        self.assertGreater(len(df), 0)
        self.assertEqual(df.columns, ['datetime', 'open', 'high', 'low', 'close', 'volume', 'token'])

    def test_save_data(self):
        df = pl.DataFrame({
            'datetime': [datetime.now()],
            'open': [1.0],
            'high': [2.0],
            'low': [0.5],
            'close': [1.5],
            'volume': [100.0],
            'token': ['BTC']
        })
        self.hyperliquid.save_data(df, 'BTC', 'parquet')
        self.assertTrue((self.data_folder / 'BTC.parquet').exists())
        self.hyperliquid.save_data(df, 'BTC', 'csv')
        self.assertTrue((self.data_folder / 'BTC.csv').exists())

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)